<a href="https://colab.research.google.com/github/yatharthanand-code/chat-emotion-detector/blob/main/Exploratory_Data_Analysis_(EDA)_on_Titanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratory Data Analysis (EDA) on the Titanic Dataset

 Guiding Questions
1. **Sex**: Did female passengers have a higher survival rate than male passengers?
2. **Passenger Class:** Did passengers traveling in higher ticket classes **pclass** have higher survival rates than those in lower classes?
3. **Age:** How did survival rates differ across age bands (such as children versus adults and seniors)?*

# Load and Inspect the Dataset
We will load the Titanic dataset using Seaborn and inspect its structure, data types, and missing values using **df.shape, df.head(), df.info(), and df.describe().**

In [12]:
import pandas as pd
import numpy as np
import seaborn as sns

# Load Titanic dataset
df = sns.load_dataset('titanic')

# Basic Inspection
print("=== Dataset Shape ===")
print(df.shape)

print("\n=== First 5 Rows ===")
print(df.head())

print("\n=== Dataset Info ===")
print(df.info())

print("\n=== Statistical Summary ===")
print(df.describe(include='all'))

=== Dataset Shape ===
(891, 15)

=== First 5 Rows ===
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  

=== Dataset Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtyp

# Data Cleaning and Handling Missing Values
Based on our inspection, we will clean the missing values using robust strategies instead of blindly dropping rows:
- **age**: We fill missing values with the median age of passengers within the same passenger class pclass and sex sex. This preserves age distributions across different demographics.
- **embarked / embark_town**: We fill the missing values with the most common value mode.
- **deck**: This column has an excessive number of missing values and will be dropped.
- **Duplicates**: We will check for and document any duplicate rows.

In [13]:
# Handle missing values for age using group-specific median
df["age"] = df["age"].fillna(df.groupby(["pclass", "sex"])["age"].transform("median"))

# Handle missing values for embarked using mode
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
if "embark_town" in df.columns:
    df["embark_town"] = df["embark_town"].fillna(df["embark_town"].mode()[0])

# Drop deck due to extremely high proportion of missing values
df = df.drop(columns=["deck"], errors='ignore')

# Check for duplicates
print("Duplicate rows found:", df.duplicated().sum())

# Confirm remaining missing values
print("\nRemaining missing values per column:")
print(df.isnull().sum())

Duplicate rows found: 118

Remaining missing values per column:
survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    0
alive          0
alone          0
dtype: int64


# Compare Survival Rates to Answer Guiding Questions
Now, we group and aggregate our cleaned data to answer the core analytical questions using real numbers.

### Question 1: Did female passengers have a higher survival rate than male passengers?
### Question 2: Did passengers traveling in higher ticket classes (`pclass`) have higher survival rates?
### Question 3: How did survival rates differ across age bands (children, teens, adults, middle-aged, seniors)?

In [14]:
# 1. Survival Rate by Sex
print("=== Survival Rate by Sex ===")
sex_survival = df.groupby("sex")["survived"].mean().round(2)
print(sex_survival)
print("\n")

# 2. Survival Rate by Ticket Class
print("=== Survival Rate by Class ===")
class_survival = df.groupby("pclass")["survived"].agg(["mean", "count"]).round(2)
print(class_survival)
print("\n")

# 3. Survival Rate by Age Band
df["age_band"] = pd.cut(df["age"], bins=[0, 12, 18, 40, 60, 100],
                        labels=["Child", "Teen", "Adult", "Middle-aged", "Senior"])

print("=== Survival Rate by Age Band ===")
age_survival = df.groupby("age_band", observed=True)["survived"].mean().round(2)
print(age_survival)

=== Survival Rate by Sex ===
sex
female    0.74
male      0.19
Name: survived, dtype: float64


=== Survival Rate by Class ===
        mean  count
pclass             
1       0.63    216
2       0.47    184
3       0.24    491


=== Survival Rate by Age Band ===
age_band
Child          0.58
Teen           0.43
Adult          0.36
Middle-aged    0.39
Senior         0.23
Name: survived, dtype: float64
